# Section V-B-3: Multi-shot OTEM under Varying Fault Occurrence Probabilities
**Fig. 16** — test pass rate and overhead for m/n configurations.
Fixed: toggle probability = 1%.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # qatg package lives in ../qatg/
sys.path.insert(0, os.path.abspath('.'))    # otem.py, fault_simulators.py live here

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit.circuit.library import QFT, GraphState, QuantumVolume, UnitaryGate
from qiskit.synthesis.one_qubit import OneQubitEulerDecomposer
import qiskit.circuit.library as qGate

from qatg import QATG, QATGFault
from fault_simulators import (Transient_Faulty_Backend, Permanent_Faulty_Backend,
                               get_counts_from_mem, get_em_counts_from_mem)

In [ ]:
_sx_decomposer = OneQubitEulerDecomposer('U')
class mySXFault(QATGFault):
    def __init__(self, fgate):
        super().__init__(qGate.SXGate, 0, "gateType: SX, qubits: 0")
        theta, phi, lam = _sx_decomposer.angles(fgate)
        self.fgate = qGate.UGate(theta, phi, lam)
    def createOriginalGate(self):
        return qGate.SXGate()
    def createFaultyGate(self, faultfreeGate):
        return self.fgate

In [ ]:
def get_depolarizing_backend(error_rate_1q, error_rate_2q=None):
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_1q, 1), ['sx', 'rz', 'u'])
    if error_rate_2q:
        nm.add_all_qubit_quantum_error(depolarizing_error(error_rate_2q, 2), ['cx', 'ecr', 'unitary'])
    return AerSimulator(noise_model=nm)

In [ ]:
faults     = np.load('data/faults.npy', allow_pickle=True)
ecr_faults = np.load('data/ecr_fault_list.npy', allow_pickle=True)
fault_id   = 6
sx_fault   = faults[fault_id]
print(f"Loaded {len(faults)} fault models  |  using fault_id={fault_id}")

In [ ]:
nqc        = 29
test_datas = [QuantumCircuit().from_qasm_file(f'test_circuits/test_circuit_{i}.qasm')
              for i in range(nqc)]
print(f"Loaded {len(test_datas)} test circuits")

In [ ]:
def correct_rate(counts, nqb=1):
    key = '0' * nqb
    tot = sum(counts.values())
    return counts.get(key, 0) / tot if tot > 0 else 0.0

In [ ]:
# (m=OT shots, n=alg shots, bidirectional, label)
CONFIGS = [
    (1,   1, False, 'm1n1'),
    (1,   1, True,  'm1n1b'),
    (3,   3, False, 'm3n3'),
    (3,   3, True,  'm3n3b'),
    (9,   9, False, 'm9n9'),
    (9,   9, True,  'm9n9b'),
    (100, 9, False, 'm100n9'),
    (100, 9, True,  'm100n9b'),
]

## Setup fixed backend

In [ ]:
SHOTS     = 50000
SWAP_RATE = 0.01
NOISE_1Q  = 0.0005
ff_b = get_depolarizing_backend(NOISE_1Q)
pf_b = Permanent_Faulty_Backend(mySXFault(sx_fault), ff_b,
                                basis_gates=['sx','rz'], num_qubits=1)

best_tid, best_rate = 0, 0.0
for tid, qc in enumerate(test_datas):
    mem  = pf_b.run(qc, shots=10000).get_memory()
    rate = sum(1 for s in mem if '1' in s) / len(mem)
    if rate > best_rate:
        best_rate, best_tid = rate, tid
test_qc = test_datas[best_tid]
print(f"Test circuit: {best_tid}")

## Sweep fault occurrence probability (x-axis)

In [ ]:
fault_occ_probs = [0.0, 0.01, 0.02, 0.03, 0.05, 0.07, 0.10]
results  = {cfg[3]: {'pass_rate': [], 'overhead': []} for cfg in CONFIGS}
ff_rates, ft_rates = [], []

for fop in fault_occ_probs:
    tf_b = Transient_Faulty_Backend(pf_b, ff_b, swap_rate=fop)

    ff_mem = ff_b.run(test_qc, shots=SHOTS, memory=True).result().get_memory()
    ft_mem = tf_b.run(test_qc, shots=SHOTS)[0]
    ff_rates.append(correct_rate(get_counts_from_mem(ff_mem)))
    ft_rates.append(correct_rate(get_counts_from_mem(ft_mem)))

    for (m, n, bidir, label) in CONFIGS:
        repeat   = max(SHOTS // (m + n), 1)
        em_mem   = tf_b.run_single_em(test_qc, test_qc,
                                      ot_shots=m, alg_shots=n, repeat=repeat)
        em_cnt   = get_em_counts_from_mem(em_mem, shots_ot=m,
                                          ot_pass_th=(m+1)//2, shots_alg=n,
                                          bidirectional_check=bidir)
        n_sel    = sum(em_cnt.values())
        total    = repeat * (m + n)
        results[label]['pass_rate'].append(correct_rate(em_cnt))
        results[label]['overhead'].append(min(total / n_sel, 9.0) if n_sel > 0 else 9.0)
    print(f"fop={fop:.3f} done")

## Plot — Fig. 16

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = plt.cm.tab10(np.linspace(0, 1, len(CONFIGS)))
ax1.plot(fault_occ_probs, ff_rates, '-',  color='tab:blue',   label='Fault Free', lw=2)
ax1.plot(fault_occ_probs, ft_rates, '--', color='tab:orange',  label='Faulty',     lw=2)
ax2.axhline(1.0, color='tab:blue', lw=2, label='Fault Free')
for (m,n,b,lbl), col in zip(CONFIGS, colors):
    ax1.plot(fault_occ_probs, results[lbl]['pass_rate'], '--o', color=col, label=lbl, ms=5)
    ax2.plot(fault_occ_probs, results[lbl]['overhead'],  '--o', color=col, label=lbl, ms=5)
for ax in (ax1, ax2):
    ax.set_xlabel('fault occurrence probability'); ax.legend(fontsize=8, ncol=2)
ax1.set_ylabel('Test Pass Rate'); ax2.set_ylabel('Overhead')
ax1.set_title('(a) Test pass rate'); ax2.set_title('(b) Overhead')
plt.suptitle('Fig. 16: Multi-shot OTEM under varying fault occurrence probabilities')
plt.tight_layout()
plt.savefig('fig16_multishot_fault_occurrence.pdf', bbox_inches='tight')
plt.show()